In [1]:
import h5py
import numpy as np
import pickle
import torch.nn.functional as F


In [2]:
base_path = "/srv/nfs-data/sisko/storage/fMRI_maas_2023"
math_path = base_path + "/Beta2hem.mat"
saved_path = "/srv/nfs-data/sisko/matteoc/maas_2023"

In [3]:
import re

def soundname_to_wav(sound_name: str) -> str:
    """
    Converte una stringa tipo:
    'stim145_cat04_music_exemp01'
    in:
    's2_music_1.wav'
    """
    m = re.match(r"stim\d+_cat\d+_([a-zA-Z]+)_exemp(\d+)", sound_name)
    if m is None:
        raise ValueError(f"Formato inatteso: {sound_name}")

    category = m.group(1).lower()
    exemp = int(m.group(2))

    return f"s2_{category}_{exemp}.wav"


In [4]:
with open(saved_path + "/pooled_data.pkl", "rb") as f:
    pooled_data = pickle.load(f)

roi = 'PT2hem'
cv = 'CV2'

entry = pooled_data[roi][cv]
print(entry.keys())

print("ROI:", roi)
print("CV:", cv)

print("X_train shape:", entry["X_train"].shape if entry["X_train"] is not None else None)
print("X_test shape:", entry["X_test"].shape if entry["X_test"] is not None else None)
print("train_sounds shape:", entry["train_sounds"].shape if hasattr(entry["train_sounds"], "shape") else type(entry["train_sounds"]))
print("test_sounds shape:", entry["test_sounds"].shape if hasattr(entry["test_sounds"], "shape") else type(entry["test_sounds"]))

print("train_sounds preview:", entry["train_sounds"][:10] if entry["train_sounds"] is not None else None)
print("test_sounds preview:", entry["test_sounds"][:10] if entry["test_sounds"] is not None else None)


dict_keys(['X_train', 'X_test', 'train_sounds', 'test_sounds', 'subjects_train', 'subjects_test', 'valid_subjects'])
ROI: PT2hem
CV: CV2
X_train shape: (1080, 1024)
X_test shape: (360, 1024)
train_sounds shape: (1080,)
test_sounds shape: (360,)
train_sounds preview: [ 4  5  6  7  8  9 10 13 14 15]
test_sounds preview: [ 1  2  3 11 12 23 25 27 28 36]


In [5]:
roi_pooled_train = []
roi_pooled_test = []
sound_pool_train_idx = []
sound_pool_test_idx = []

roi_list = ['HG2hem', 'PT2hem', 'PP2hem', 'mSTG2hem', 'pSTG2hem', 'aSTG2hem']

for reg in roi_list:
    entry = pooled_data[reg][cv]
    roi_pooled_train.append(entry["X_train"])
    roi_pooled_test.append(entry["X_test"])
    if reg == "HG2hem":
        sound_pool_train_idx.append(entry["train_sounds"])
        sound_pool_test_idx.append(entry["test_sounds"])

roi_pooled_train = np.stack(roi_pooled_train, axis=1)
roi_pooled_test = np.stack(roi_pooled_test, axis=1)
sound_pool_train_idx = np.array(sound_pool_train_idx)[0]
sound_pool_test_idx = np.array(sound_pool_test_idx)[0]

print(roi_pooled_train.shape)
print(roi_pooled_test.shape)
print(sound_pool_train_idx.shape)
print(sound_pool_test_idx.shape)

(1080, 6, 1024)
(360, 6, 1024)
(1080,)
(360,)


In [6]:
import os
import numpy as np

wav_dir = "/srv/nfs-data/sisko/storage/fMRI_maas_2023/wav"

name_sounds = np.load("/home/matteoc/brainSounds/maas_data/SoundNames.npy")

idx_train = sound_pool_train_idx.astype(int) - 1
idx_test  = sound_pool_test_idx.astype(int) - 1

train_wav_paths = [
    os.path.join(wav_dir, soundname_to_wav(name_sounds[i]))
    for i in idx_train
]

test_wav_paths = [
    os.path.join(wav_dir, soundname_to_wav(name_sounds[i]))
    for i in idx_test
]

## Stable Audio

In [7]:
import torch
from diffusers import (
    StableAudioPipeline,
    EDMDPMSolverMultistepScheduler,
    AutoencoderOobleck,
    StableAudioDiTModel,
    StableAudioProjectionModel,
)
from transformers import T5EncoderModel, T5Tokenizer


device = "cuda:5" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device.startswith("cuda") else torch.float32

repo_id = "stabilityai/stable-audio-open-1.0"

pipe = StableAudioPipeline.from_pretrained(
    repo_id,
    torch_dtype=dtype,
).to(device)

print("Transformer model class:", pipe.transformer.__class__.__name__)
print("Transformer cross_attention_dim:", getattr(pipe.transformer.config, "cross_attention_dim", None))
print("Transformer attention_head_dim:", getattr(pipe.transformer.config, "attention_head_dim", None))
print("Transformer num_attention_heads:", getattr(pipe.transformer.config, "num_attention_heads", None))
print("Transformer in_channels:", getattr(pipe.transformer.config, "in_channels", None))
print("Transformer out_channels:", getattr(pipe.transformer.config, "out_channels", None))

print("Text encoder hidden size:", pipe.text_encoder.config.d_model)
print("Tokenizer model max length:", pipe.tokenizer.model_max_length)

print("Projection model class:", pipe.projection_model.__class__.__name__)

print("VAE model class:", pipe.vae.__class__.__name__)
print("VAE sampling_rate:", getattr(pipe.vae, "sampling_rate", None))
print("VAE hop_length:", getattr(pipe.vae, "hop_length", None))
print("VAE audio_channels:", getattr(pipe.vae.config, "audio_channels", None))
print("VAE latent_dim:", getattr(pipe.vae.config, "latent_dim", None))

print("Pipeline vae_scale_factor:", getattr(pipe, "vae_scale_factor", None))

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

/home/matteoc/miniconda3/envs/huggin/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Transformer model class: StableAudioDiTModel
Transformer cross_attention_dim: 768
Transformer attention_head_dim: 64
Transformer num_attention_heads: 24
Transformer in_channels: 64
Transformer out_channels: 64
Text encoder hidden size: 768
Tokenizer model max length: 128
Projection model class: StableAudioProjectionModel
VAE model class: AutoencoderOobleck
VAE sampling_rate: 44100
VAE hop_length: 2048
VAE audio_channels: 2
VAE latent_dim: None
Pipeline vae_scale_factor: None


In [21]:
import torch
import tqdm

from diffusers import StableAudioPipeline
from diffusers.pipelines.stable_audio.pipeline_stable_audio import get_1d_rotary_pos_embed


@torch.no_grad()
def extract_stable_audio_conditioning(
    pipe: StableAudioPipeline,
    prompts,
    audio_start_in_s=0.0,
    audio_end_in_s=10.0,
    batch_size=8,
    device=None,
    num_inference_steps=100,
    guidance_scale=7.0,
    negative_prompt=None,
    num_waveforms_per_prompt=1,
    eta=0.0,
    generator=None,
    latents=None,
    initial_audio_waveforms=None,
    initial_audio_sampling_rate=None,
    return_all_noise_preds=False,
    output_type="latent",
):
    """
    Restituisce:
      - raw_t5_hidden_states
      - attention_mask
      - projected_prompt_embeds
      - seconds_start_hidden_states
      - seconds_end_hidden_states
      - text_audio_duration_embeds
      - audio_duration_embeds
      - raw_noise_pred
      - guided_noise_pred
      - final_latents

    Note:
      - raw_noise_pred: output diretto del transformer
      - guided_noise_pred: dopo classifier-free guidance (se guidance_scale > 1)
      - se return_all_noise_preds=True, vengono ritornati tutti i noise_pred di tutti i timestep
    """

    if device is None:
        device = pipe._execution_device if hasattr(pipe, "_execution_device") else pipe.device

    pipe.text_encoder.eval()
    pipe.projection_model.eval()
    pipe.transformer.eval()

    do_classifier_free_guidance = guidance_scale > 1.0

    all_raw_t5 = []
    all_attn = []
    all_proj = []
    all_sec_start = []
    all_sec_end = []
    all_text_audio_duration = []
    all_audio_duration = []

    all_raw_noise_pred = []
    all_guided_noise_pred = []
    all_final_latents = []

    # Per controllare la lunghezza audio come nel __call__
    downsample_ratio = pipe.vae.hop_length
    max_audio_length_in_s = (
        pipe.transformer.config.sample_size * downsample_ratio / pipe.vae.config.sampling_rate
    )

    if audio_end_in_s is None:
        audio_end_in_s = max_audio_length_in_s

    if audio_end_in_s - audio_start_in_s > max_audio_length_in_s:
        raise ValueError(
            f"The total audio length requested ({audio_end_in_s - audio_start_in_s}s) "
            f"is longer than the model maximum possible length ({max_audio_length_in_s})."
        )

    waveform_length = int(pipe.transformer.config.sample_size)

    for start in tqdm.tqdm(range(0, len(prompts), batch_size)):
        batch_prompts = prompts[start:start + batch_size]
        curr_bs = len(batch_prompts)

        # Gestione negative_prompt per batch
        if negative_prompt is None:
            batch_negative_prompt = None
        elif isinstance(negative_prompt, str):
            batch_negative_prompt = [negative_prompt] * curr_bs
        else:
            batch_negative_prompt = negative_prompt[start:start + curr_bs]

        #
        # 1) TOKENIZZAZIONE + T5 RAW
        #
        text_inputs = pipe.tokenizer(
            batch_prompts,
            padding="max_length",
            max_length=pipe.tokenizer.model_max_length,
            truncation=True,
            return_tensors="pt",
        )
        input_ids = text_inputs.input_ids.to(device)
        attention_mask = text_inputs.attention_mask.to(device)

        raw_t5_hidden = pipe.text_encoder(
            input_ids,
            attention_mask=attention_mask,
        )[0]

        #
        # 2) PROIEZIONE TESTO
        #
        projected_prompt_embeds = pipe.projection_model(
            text_hidden_states=raw_t5_hidden
        ).text_hidden_states

        projected_prompt_embeds = (
            projected_prompt_embeds
            * attention_mask.unsqueeze(-1).to(projected_prompt_embeds.dtype)
        )

        #
        # 3) PROMPT EMBEDS "VERI" DELLA PIPELINE
        #    Qui usiamo encode_prompt, così seguiamo la logica del __call__
        #
        prompt_embeds = pipe.encode_prompt(
            prompt=batch_prompts,
            device=device,
            do_classifier_free_guidance=do_classifier_free_guidance,
            negative_prompt=batch_negative_prompt,
            prompt_embeds=None,
            negative_prompt_embeds=None,
            attention_mask=None,
            negative_attention_mask=None,
        )

        #
        # 4) DURATION CONDITIONING
        #
        sec_start, sec_end = pipe.encode_duration(
            audio_start_in_s=audio_start_in_s,
            audio_end_in_s=audio_end_in_s,
            device=device,
            do_classifier_free_guidance=(
                do_classifier_free_guidance and (batch_negative_prompt is not None)
            ),
            batch_size=curr_bs,
        )

        #
        # 5) CONCAT CONDITIONING COME NEL __call__
        #
        text_audio_duration_embeds = torch.cat(
            [prompt_embeds, sec_start, sec_end],
            dim=1
        )
        audio_duration_embeds = torch.cat([sec_start, sec_end], dim=2)

        # Caso speciale: CFG senza negative prompt esplicito
        if do_classifier_free_guidance and batch_negative_prompt is None:
            negative_text_audio_duration_embeds = torch.zeros_like(
                text_audio_duration_embeds,
                device=text_audio_duration_embeds.device
            )
            text_audio_duration_embeds = torch.cat(
                [negative_text_audio_duration_embeds, text_audio_duration_embeds],
                dim=0
            )
            audio_duration_embeds = torch.cat(
                [audio_duration_embeds, audio_duration_embeds],
                dim=0
            )

        #
        # 6) DUPLICAZIONE PER num_waveforms_per_prompt
        #
        bs_embed, seq_len, hidden_size = text_audio_duration_embeds.shape

        text_audio_duration_embeds = text_audio_duration_embeds.repeat(
            1, num_waveforms_per_prompt, 1
        )
        text_audio_duration_embeds = text_audio_duration_embeds.view(
            bs_embed * num_waveforms_per_prompt, seq_len, hidden_size
        )

        audio_duration_embeds = audio_duration_embeds.repeat(
            1, num_waveforms_per_prompt, 1
        )
        audio_duration_embeds = audio_duration_embeds.view(
            bs_embed * num_waveforms_per_prompt,
            -1,
            audio_duration_embeds.shape[-1]
        )

        #
        # 7) TIMESTEPS
        #
        pipe.scheduler.set_timesteps(num_inference_steps, device=device)
        timesteps = pipe.scheduler.timesteps

        #
        # 8) LATENTS
        #
        num_channels_vae = pipe.transformer.config.in_channels
        batch_latent_size = curr_bs * num_waveforms_per_prompt

        batch_latents = pipe.prepare_latents(
            batch_latent_size,
            num_channels_vae,
            waveform_length,
            text_audio_duration_embeds.dtype,
            device,
            generator,
            latents=None if latents is None else latents[start:start + curr_bs],
            initial_audio_waveforms=(
                None if initial_audio_waveforms is None
                else initial_audio_waveforms[start:start + curr_bs]
            ),
            num_waveforms_per_prompt=num_waveforms_per_prompt,
            audio_channels=pipe.vae.config.audio_channels,
        )

        #
        # 9) EXTRA STEP KWARGS
        #
        extra_step_kwargs = pipe.prepare_extra_step_kwargs(generator, eta)

        #
        # 10) ROTARY EMBEDDING
        #
        rotary_embedding = get_1d_rotary_pos_embed(
            pipe.rotary_embed_dim,
            batch_latents.shape[2] + audio_duration_embeds.shape[1],
            use_real=True,
            repeat_interleave_real=False,
        )

        #
        # 11) DENOISING LOOP
        #
        batch_raw_noise_preds = []
        batch_guided_noise_preds = []

        for i, t in enumerate(timesteps):
            latent_model_input = (
                torch.cat([batch_latents] * 2) if do_classifier_free_guidance else batch_latents
            )
            latent_model_input = pipe.scheduler.scale_model_input(latent_model_input, t)

            raw_noise_pred = pipe.transformer(
                latent_model_input,
                t.unsqueeze(0),
                encoder_hidden_states=text_audio_duration_embeds,
                global_hidden_states=audio_duration_embeds,
                rotary_embedding=rotary_embedding,
                return_dict=False,
            )[0]

            guided_noise_pred = raw_noise_pred
            if do_classifier_free_guidance:
                noise_pred_uncond, noise_pred_text = raw_noise_pred.chunk(2)
                guided_noise_pred = noise_pred_uncond + guidance_scale * (
                    noise_pred_text - noise_pred_uncond
                )

            batch_latents = pipe.scheduler.step(
                guided_noise_pred, t, batch_latents, **extra_step_kwargs
            ).prev_sample

            if return_all_noise_preds:
                batch_raw_noise_preds.append(raw_noise_pred.cpu())
                batch_guided_noise_preds.append(guided_noise_pred.cpu())

        # Se non vuoi tutti i timestep, salvo solo l'ultimo
        if return_all_noise_preds:
            all_raw_noise_pred.append(torch.stack(batch_raw_noise_preds, dim=1))       # [B, T, ...] o [2B, T, ...]
            all_guided_noise_pred.append(torch.stack(batch_guided_noise_preds, dim=1)) # [B, T, ...]
        else:
            all_raw_noise_pred.append(raw_noise_pred.cpu())
            all_guided_noise_pred.append(guided_noise_pred.cpu())

        #
        # 12) SALVATAGGIO OUTPUT INTERMEDI
        #
        all_raw_t5.append(raw_t5_hidden.cpu())
        all_attn.append(attention_mask.cpu())
        all_proj.append(projected_prompt_embeds.cpu())
        all_sec_start.append(sec_start.cpu())
        all_sec_end.append(sec_end.cpu())
        all_text_audio_duration.append(text_audio_duration_embeds.cpu())
        all_audio_duration.append(audio_duration_embeds.cpu())
        all_final_latents.append(batch_latents.cpu())

    out = {
        "raw_t5_hidden_states": torch.cat(all_raw_t5, dim=0),
        "attention_mask": torch.cat(all_attn, dim=0),
        "projected_prompt_embeds": torch.cat(all_proj, dim=0),
        "seconds_start_hidden_states": torch.cat(all_sec_start, dim=0),
        "seconds_end_hidden_states": torch.cat(all_sec_end, dim=0),
        "text_audio_duration_embeds": torch.cat(all_text_audio_duration, dim=0),
        "audio_duration_embeds": torch.cat(all_audio_duration, dim=0),
        "raw_noise_pred": torch.cat(all_raw_noise_pred, dim=0),
        "guided_noise_pred": torch.cat(all_guided_noise_pred, dim=0),
        "final_latents": torch.cat(all_final_latents, dim=0),
    }

    if output_type == "pt":
        return out
    elif output_type == "latent":
        return out, t.unsqueeze(0), rotary_embedding, latent_model_input
    else:
        raise ValueError("output_type deve essere 'pt' oppure 'latent'")

In [22]:
prompts = [
    "a calm piano melody",
    "heavy techno beat with bass"
]

out, t, rotar, latent_model_input = extract_stable_audio_conditioning(
    pipe,
    prompts,
    audio_start_in_s=0.0,
    audio_end_in_s=1.0,
    batch_size=2,
)

  0%|          | 0/1 [00:00<?, ?it/s]/home/matteoc/miniconda3/envs/huggin/lib/python3.11/site-packages/torchsde/_brownian/brownian_interval.py:608: UserWarning: Should have tb<=t1 but got tb=500.00006103515625 and t1=500.0.
  warnings.warn(f"Should have {tb_name}<=t1 but got {tb_name}={tb} and t1={self._end}.")
/home/matteoc/miniconda3/envs/huggin/lib/python3.11/site-packages/torchsde/_brownian/brownian_interval.py:599: UserWarning: Should have ta>=t0 but got ta=0.29999998211860657 and t0=0.3.
  warnings.warn(f"Should have ta>=t0 but got ta={ta} and t0={self._start}.")
/home/matteoc/miniconda3/envs/huggin/lib/python3.11/site-packages/torchsde/_brownian/brownian_interval.py:599: UserWarning: Should have ta>=t0 but got ta=0.0 and t0=0.3.
  warnings.warn(f"Should have ta>=t0 but got ta={ta} and t0={self._start}.")
/home/matteoc/miniconda3/envs/huggin/lib/python3.11/site-packages/torchsde/_brownian/brownian_interval.py:602: UserWarning: Should have tb>=t0 but got tb=0.29999998211860657 and

In [27]:
rotar[0].shape, rotar[1].shape, latent_model_input.shape

(torch.Size([1025, 32]), torch.Size([1025, 32]), torch.Size([4, 64, 1024]))

In [13]:
def print_out_shapes(out):
    print("\n===== OUTPUT SHAPES =====\n")

    for key, value in out.items():
        if isinstance(value, torch.Tensor):
            print(f"{key:35s} -> {tuple(value.shape)}")
        elif isinstance(value, list):
            try:
                print(f"{key:35s} -> list of {len(value)} elements, first shape: {tuple(value[0].shape)}")
            except:
                print(f"{key:35s} -> list of {len(value)} elements")
        else:
            print(f"{key:35s} -> {type(value)}")

    print("\n=========================\n")


print_out_shapes(out)


===== OUTPUT SHAPES =====

raw_t5_hidden_states                -> (2, 128, 768)
attention_mask                      -> (2, 128)
projected_prompt_embeds             -> (2, 128, 768)
seconds_start_hidden_states         -> (2, 1, 768)
seconds_end_hidden_states           -> (2, 1, 768)
text_audio_duration_embeds          -> (4, 130, 768)
audio_duration_embeds               -> (4, 1, 1536)
raw_noise_pred                      -> (4, 64, 1024)
guided_noise_pred                   -> (2, 64, 1024)
final_latents                       -> (2, 64, 1024)




In [28]:
cond = out["text_audio_duration_embeds"].to(device)
audio_duration_embeds = out["audio_duration_embeds"].to(device)

cond_small = cond[:, :30, :]  # taglio semplice

y = pipe.transformer(
    latent_model_input,
    t,
    encoder_hidden_states=cond_small,
    global_hidden_states=audio_duration_embeds,
    rotary_embedding=rotar,
    return_dict=False,
)[0]

print(y.shape)

torch.Size([4, 64, 1024])


In [32]:
import inspect

print(inspect.signature(pipe.transformer.forward))

(hidden_states: torch.FloatTensor, timestep: torch.LongTensor = None, encoder_hidden_states: torch.FloatTensor = None, global_hidden_states: torch.FloatTensor = None, rotary_embedding: torch.FloatTensor = None, return_dict: bool = True, attention_mask: Optional[torch.LongTensor] = None, encoder_attention_mask: Optional[torch.LongTensor] = None) -> Union[torch.FloatTensor, diffusers.models.transformers.transformer_2d.Transformer2DModelOutput]


## Run Diffusion